In [14]:
# %pip install seaborn scikit_posthocs pingouin

In [ ]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [16]:
df_xexymix_reviews = pd.read_csv('./data/xexymix_raw_review_final.csv')

## 젝시믹스 리뷰

In [17]:
df_xexymix_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 2364102 entries, 0 to 2364101
Data columns (total 14 columns):
 #   Column            Dtype  
---  ------            -----  
 0   collect_date      str    
 1   post_date         str    
 2   platform          str    
 3   brand             str    
 4   search_keyword    float64
 5   content           str    
 6   source_url        str    
 7   rating            int64  
 8   helpful_count     int64  
 9   purchased_option  str    
 10  has_image         int64  
 11  user_height       str    
 12  user_weight       float64
 13  user_size         str    
dtypes: float64(2), int64(3), str(9)
memory usage: 252.5 MB


### 컬럼명 변경

In [18]:
df_xexymix_reviews = df_xexymix_reviews.rename(columns={
    'post_date': 'review_date',
    'source_url': 'review_url',
})

In [19]:
print(df_xexymix_reviews.columns.tolist())


['collect_date', 'review_date', 'platform', 'brand', 'search_keyword', 'content', 'review_url', 'rating', 'helpful_count', 'purchased_option', 'has_image', 'user_height', 'user_weight', 'user_size']


### 형 변환(datetime)

In [20]:
df_xexymix_reviews['review_date'] = pd.to_datetime(df_xexymix_reviews['review_date'])
df_xexymix_reviews['collect_date'] = pd.to_datetime(df_xexymix_reviews['collect_date'])

### 형 변환(category)

In [21]:
df_xexymix_reviews['platform'] = df_xexymix_reviews['platform'].astype('category')
df_xexymix_reviews['purchased_option'] = df_xexymix_reviews['purchased_option'].astype('category')
df_xexymix_reviews['user_size'] = df_xexymix_reviews['user_size'].astype('category')

### 결측치 처리(None)

In [22]:
(df_xexymix_reviews.isna().mean() * 100).sort_values(ascending=False)

search_keyword      100.000000
user_size            17.906165
user_weight          13.323241
user_height          10.445065
collect_date          0.000000
review_date           0.000000
platform              0.000000
brand                 0.000000
content               0.000000
review_url            0.000000
rating                0.000000
helpful_count         0.000000
purchased_option      0.000000
has_image             0.000000
dtype: float64

### 컬럼만 남기고 데이터 제거(컬럼만 유지)

In [23]:
df_xexymix_reviews['search_keyword'] = ''

In [24]:
df_xexymix_reviews['search_keyword']

0           
1           
2           
3           
4           
          ..
2364097     
2364098     
2364099     
2364100     
2364101     
Name: search_keyword, Length: 2364102, dtype: str

### 이상치 확인

In [25]:
df_xexymix_reviews['helpful_count'].describe()

count    2.364102e+06
mean     1.536748e-01
std      2.744528e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      6.110000e+02
Name: helpful_count, dtype: float64

In [26]:
df_xexymix_reviews['has_image'].value_counts()

has_image
1    2364102
Name: count, dtype: int64

### 고객 키 그룹화

In [27]:
df_xexymix_reviews['user_height'] = pd.to_numeric(df_xexymix_reviews['user_height'], errors='coerce')

In [28]:
# 구간 경계 정의
bins = [0, 139, 144, 149, 154, 159, 164, 169, 174, 179, 184, 189, float('inf')]

# 라벨 정의
user_height_group = [
    '139cm 이하',
    '140~144cm',
    '145~149cm',
    '150~154cm',
    '155~159cm',
    '160~164cm',
    '165~169cm',
    '170~174cm',
    '175~179cm',
    '180~184cm',
    '185~189cm',
    '190cm 이상'
]

df_xexymix_reviews['user_height_group'] = pd.cut(df_xexymix_reviews['user_height'], bins=bins, labels=user_height_group, right=False)

In [29]:
df_xexymix_reviews['user_height_group'].value_counts()

user_height_group
160~164cm    701185
165~169cm    504017
155~159cm    392435
170~174cm    223250
175~179cm    121961
150~154cm     66980
180~184cm     56722
185~189cm     13630
139cm 이하       3555
145~149cm      2643
190cm 이상       2155
140~144cm       645
Name: count, dtype: int64

### 고객 몸무게 그룹화

In [30]:
df_xexymix_reviews['user_weight'] = pd.to_numeric(df_xexymix_reviews['user_weight'], errors='coerce')

In [31]:
# 구간 경계 정의
bins = [0, 44, 49, 54, 59, 64, 69, 74, 79, 84, 89, float('inf')]

# 라벨 정의
user_weight_group = [
    '44kg 이하',
    '45~49kg',
    '50~54kg',
    '55~59kg',
    '60~64kg',
    '65~69kg',
    '70~74kg',
    '75~79kg',
    '80~84kg',
    '85~89kg',
    '90kg 이상'
]

df_xexymix_reviews['user_weight_group'] = pd.cut(df_xexymix_reviews['user_weight'], bins=bins, labels=user_weight_group, right=False)

In [32]:
df_xexymix_reviews['user_weight_group'].value_counts()

user_weight_group
50~54kg    553633
55~59kg    441703
45~49kg    343202
60~64kg    257635
65~69kg    144090
70~74kg    101162
75~79kg     72312
80~84kg     56889
90kg 이상     44481
85~89kg     32797
44kg 이하      1223
Name: count, dtype: int64

### 전처리

In [33]:
df_xexymix_reviews['content'] = (
    df_xexymix_reviews['content']
    .fillna('')  # 결측치 처리
    .apply(lambda x: re.sub(r'\s+', ' ', str(x)))  # 모든 공백 통합 (\n, \t 포함)
    .str.replace(r'[^가-힣a-zA-Z0-9\s]', '', regex=True)  # 특수문자 제거
    .str.strip()  # 앞뒤 공백 제거
)

### 2024년 이후 데이터 저장

In [34]:
# 데이터 분리
xexymix_homepage_review_final_s2024 = df_xexymix_reviews[
    df_xexymix_reviews['review_date'] >= '2024-01-01'
].copy()

xexymix_homepage_review_final = df_xexymix_reviews.copy()

# 삭제할 컬럼
cols_to_drop = ['brand']

# 컬럼 제거
xexymix_homepage_review_final_s2024 = xexymix_homepage_review_final_s2024.drop(columns=cols_to_drop, errors='ignore')
xexymix_homepage_review_final = xexymix_homepage_review_final.drop(columns=cols_to_drop, errors='ignore')

# 저장
import os
os.makedirs('data', exist_ok=True)
xexymix_homepage_review_final_s2024.to_csv('data/xexymix_homepage_review_final_s2024.csv', index=False, encoding='utf-8-sig')
xexymix_homepage_review_final.to_csv('data/xexymix_homepage_review_final.csv', index=False, encoding='utf-8-sig')

In [35]:
df = xexymix_homepage_review_final_s2024.copy()
df.info()

<class 'pandas.DataFrame'>
Index: 1300197 entries, 0 to 2364074
Data columns (total 15 columns):
 #   Column             Non-Null Count    Dtype         
---  ------             --------------    -----         
 0   collect_date       1300197 non-null  datetime64[us]
 1   review_date        1300197 non-null  datetime64[us]
 2   platform           1300197 non-null  category      
 3   search_keyword     1300197 non-null  str           
 4   content            1300197 non-null  str           
 5   review_url         1300197 non-null  str           
 6   rating             1300197 non-null  int64         
 7   helpful_count      1300197 non-null  int64         
 8   purchased_option   1300197 non-null  category      
 9   has_image          1300197 non-null  int64         
 10  user_height        1146663 non-null  float64       
 11  user_weight        1126670 non-null  float64       
 12  user_size          1061485 non-null  category      
 13  user_height_group  1146663 non-null  catego

### 중복 확인 및 제거

In [37]:
# 중복 확인
print(f"전체 행 수: {len(df):,}")
print(f"id 기준 중복 수: {df.duplicated(subset=['review_id']).sum():,}")
print(f"id 고유값 수: {df['id'].nunique():,}")

전체 행 수: 1,300,197


KeyError: Index(['review_id'], dtype='str')

In [ ]:
# 중복 제거 (id 기준)
df = df.drop_duplicates(subset=['id'])
df = df.reset_index(drop=True)

print(f"중복 제거 후 행 수: {len(df):,}")

중복 제거 후 행 수: 510,072


In [ ]:
# review_url → product_url 컬럼명 변경
df = df.rename(columns={'review_url': 'product_url'})

print(df.columns.tolist())

['id', 'collect_date', 'review_date', 'platform', 'search_keyword', 'content', 'product_url', 'rating', 'helpful_count', 'purchased_option', 'has_image', 'user_height', 'user_weight', 'user_size', 'user_height_group', 'user_weight_group']


In [ ]:
df['purchased_option'].value_counts().head(20)

purchased_option
[]                                                                              33493
[{'name': '사이즈', 'value': 'M(55반~66)'}]                                         30287
[{'name': '색상', 'value': '블랙'}]                                                 25625
[{'name': '사이즈', 'value': 'S(44~55)'}]                                          22467
[{'name': '사이즈', 'value': 'L(66반~77)'}]                                         15172
[{'name': '색상', 'value': '블랙'}, {'name': '사이즈', 'value': 'L'}]                  11783
[{'name': '색상', 'value': '블랙'}, {'name': '사이즈', 'value': 'M(55반~66)'}]          10284
[{'name': '색상', 'value': '화이트'}]                                                 9241
[{'name': '색상', 'value': '블랙'}, {'name': '사이즈', 'value': 'S(44~55)'}]            6283
[{'name': '색상', 'value': '블랙'}, {'name': '사이즈', 'value': 'XL'}]                  5601
[{'name': '색상', 'value': '블랙'}, {'name': '사이즈', 'value': 'L(66반~77)'}]           5469
[{'name': '사이즈', 'value': '240'}]    

In [ ]:
def parse_purchased_option(val):
    try:
        options = ast.literal_eval(str(val))
        if not options:
            return ''
        return ' / '.join(f"{opt['name']}:{opt['value']}" for opt in options)
    except:
        return ''

df['purchased_option'] = df['purchased_option'].astype(str).apply(parse_purchased_option)

df['purchased_option'].value_counts().head(10)

purchased_option
                         33493
사이즈:M(55반~66)            30287
색상:블랙                    25625
사이즈:S(44~55)             22467
사이즈:L(66반~77)            15172
색상:블랙 / 사이즈:L            11783
색상:블랙 / 사이즈:M(55반~66)    10284
색상:화이트                    9241
색상:블랙 / 사이즈:S(44~55)      6283
색상:블랙 / 사이즈:XL            5601
Name: count, dtype: int64

In [ ]:
df['purchased_option']

0                                    색상:화이트 / 사이즈:L
1                                   색상:블랙 / 사이즈:S/M
2                                     색상:블랙 / 사이즈:L
3                                    색상:화이트 / 사이즈:L
4                                  색상:화이트 / 사이즈:S/M
                            ...                    
510067                        색상:핑크 / 사이즈:M(55반~66)
510068    상의(XGFVT20J3):베이지 S / 하의(XGFSK20J3):베이지 S
510069                       색상:베이지 / 사이즈:L(66반~77)
510070                        색상:핑크 / 사이즈:M(55반~66)
510071                       색상:베이지 / 사이즈:XL(77~88)
Name: purchased_option, Length: 510072, dtype: str

In [ ]:
# df 추출
df.to_csv('./data/xexymix_homepage_review_s2024_260425.csv', index=False, encoding='utf-8-sig')